In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
import pandas as pd
import numpy as np
nums_rows=100000
user_pool=[]
session_pool=[]
np.random.seed(5)
event_types=['view','click','scroll','cart_add']
for i in range(1000,6000):
    user_id=f'user_{i}'
    user_pool.append(user_id)
for i in range(0,30000):
    session_id=f'session_{i}'
    session_pool.append(session_id)
random_user_pool=np.random.choice(user_pool,size=nums_rows)
random_event_type=np.random.choice(event_types,size=nums_rows)
random_session_id=np.random.choice(session_pool,size=nums_rows)
random_dates=pd.date_range(start='2026-01-01',end='2026-12-31',periods=nums_rows)
print(random_session_id)
    

['session_23027' 'session_14557' 'session_2258' ... 'session_9592'
 'session_4849' 'session_6123']


In [3]:
main_table=pd.DataFrame({'user_id':random_user_pool,'session_id':random_session_id,'event_type':random_event_type,'timestamp':random_dates})
duplicated_table=main_table.head(5000)
duplicated_table
merged_table=pd.concat([main_table,duplicated_table])
merged_table=merged_table.reset_index(drop=True)
merged_table

,user_id,session_id,event_type,timestamp
0,user_3915,session_23027,scroll,2026-01-01 00:00:00.000000000
1,user_3254,session_14557,scroll,2026-01-01 00:05:14.499144991
2,user_5079,session_2258,click,2026-01-01 00:10:28.998289982
3,user_2725,session_8323,scroll,2026-01-01 00:15:43.497434974
4,user_4046,session_8410,cart_add,2026-01-01 00:20:57.996579965
...,...,...,...,...
104995,user_4930,session_28038,scroll,2026-01-19 04:22:03.229232292
104996,user_4637,session_4752,cart_add,2026-01-19 04:27:17.728377283
104997,user_4061,session_4123,scroll,2026-01-19 04:32:32.227522275
104998,user_2711,session_15301,cart_add,2026-01-19 04:37:46.726667266


In [4]:
user_id=['user_9999','user_9999','user_9999','user_9999','user_9999']
event_type=['click','click','click','click','click']
session_id=['SESS_99999','SESS_99999','SESS_99999','SESS_99999','SESS_99999']
timestamp=['2026-01-01 00:05:14.499144991','2026-01-01 00:05:14.499144991','2026-01-01 00:05:14.499144991','2026-01-01 00:05:14.499144991','2026-01-01 00:05:14.499144991']
df_bot=pd.DataFrame({'user_id':user_id,'session_id':session_id,'event_type':event_type,'timestamp':timestamp})
exploded_df_bot=df_bot.loc[np.repeat(df_bot.index,50)].reset_index(drop=True)
exploded_df_bot['timestamp']=pd.to_datetime(exploded_df_bot['timestamp'])
final_merged_table=pd.concat([merged_table,exploded_df_bot])
final_merged_table=final_merged_table.reset_index(drop=True)
final_merged_table

,user_id,session_id,event_type,timestamp
0,user_3915,session_23027,scroll,2026-01-01 00:00:00.000000000
1,user_3254,session_14557,scroll,2026-01-01 00:05:14.499144991
2,user_5079,session_2258,click,2026-01-01 00:10:28.998289982
3,user_2725,session_8323,scroll,2026-01-01 00:15:43.497434974
4,user_4046,session_8410,cart_add,2026-01-01 00:20:57.996579965
...,...,...,...,...
105245,user_9999,SESS_99999,click,2026-01-01 00:05:14.499144991
105246,user_9999,SESS_99999,click,2026-01-01 00:05:14.499144991
105247,user_9999,SESS_99999,click,2026-01-01 00:05:14.499144991
105248,user_9999,SESS_99999,click,2026-01-01 00:05:14.499144991


In [5]:
final_merged_table.to_parquet('marged_table.parquet')

In [6]:
import duckdb as db
query_test=db.query("""
with main_table as(
select user_id,session_id,event_type,timestamp,lag(timestamp,19) over(partition by user_id order by timestamp asc) as prev_time
from final_merged_table
)
select user_id,session_Id,event_type
from main_table 
where epoch(timestamp-prev_time)<10

""").df()
query_test

,user_id,session_id,event_type
0,user_9999,SESS_99999,click
1,user_9999,SESS_99999,click
2,user_9999,SESS_99999,click
3,user_9999,SESS_99999,click
4,user_9999,SESS_99999,click
...,...,...,...
226,user_9999,SESS_99999,click
227,user_9999,SESS_99999,click
228,user_9999,SESS_99999,click
229,user_9999,SESS_99999,click


In [12]:
import duckdb as db 
new_query=db.query("""
with main_table as(
select user_id,cast(timestamp as date)-(row_number() over(partition by user_id order by timestamp asc)* interval '1 day') as streak,timestamp
from final_merged_table
)
select user_id,count(streak),max(timestamp),min(timestamp)
from main_table
group by user_id,streak
""").df()
new_query

,user_id,count(streak),"max(""timestamp"")","min(""timestamp"")"
0,user_4505,1,2026-06-21 01:20:58.320583204,2026-06-21 01:20:58.320583204
1,user_4505,1,2026-10-02 09:41:58.045180448,2026-10-02 09:41:58.045180448
2,user_4528,1,2026-06-02 08:29:21.713617136,2026-06-02 08:29:21.713617136
3,user_4528,2,2026-06-15 06:26:44.680046800,2026-06-14 14:01:18.840788406
4,user_4550,1,2026-05-02 12:37:48.454684546,2026-05-02 12:37:48.454684546
...,...,...,...,...
99111,user_9999,1,2026-01-01 00:05:14.499144991,2026-01-01 00:05:14.499144991
99112,user_9999,1,2026-01-01 00:05:14.499144991,2026-01-01 00:05:14.499144991
99113,user_9999,1,2026-01-01 00:05:14.499144991,2026-01-01 00:05:14.499144991
99114,user_9999,1,2026-01-01 00:05:14.499144991,2026-01-01 00:05:14.499144991


In [15]:
import duckdb as db
query2=db.query("""
select user_id,session_id,count(*) as count
from final_merged_table
group by user_id,session_id
order by count(*) desc
limit 10
""").df()
query2 

,user_id,session_id,count
0,user_9999,SESS_99999,250
1,user_4627,session_28717,3
2,user_5943,session_17224,3
3,user_3194,session_11596,3
4,user_3499,session_20987,2
5,user_3437,session_23665,2
6,user_5373,session_29991,2
7,user_5776,session_28772,2
8,user_4496,session_12331,2
9,user_5468,session_13706,2


In [18]:
import duckdb as db
query3=db.query("""
select event_type,count(*) as count
from final_merged_table
group by event_type
""").df()
query3

,event_type,count
0,click,26429
1,cart_add,26404
2,view,26348
3,scroll,26069


,user_id,session_id,min_view,min_cart_add,diff
0,user_4221,session_18626,2026-07-31 15:20:09.288092880,2026-11-16 05:01:28.020880208,9.294079e+06
1,user_1170,session_10929,2026-08-25 01:00:11.556115560,2026-11-19 01:41:24.348843488,7.432873e+06
2,user_1910,session_15335,2026-09-02 01:01:20.676806768,2026-10-16 14:29:39.209792096,3.850099e+06


In [46]:
import duckdb as db
query5=db.query("""
with new as(
select user_id,(cast(sum(case when event_type='click' then 1 else 0 end) as numeric)/nullif(sum(case when event_type='scroll' then 1 else 0 end),0)
) as click_scroll_ratio,count(*) as count
from final_merged_table
group by user_id
)
select user_id,click_scroll_ratio
from new
where click_scroll_ratio>5.0 and count>10
""").df()
query5

,user_id,click_scroll_ratio
0,user_4297,7.0
1,user_5573,7.0
2,user_3481,5.5
3,user_1486,8.0
4,user_1621,5.5
...,...,...
71,user_1778,6.0
72,user_2054,6.0
73,user_4534,7.0
74,user_3988,6.0
